In [10]:
!pip install scikit-learn pandas nltk

In [11]:
import pandas as pd
import numpy as np
import nltk
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

nltk.download('stopwords')
from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
COLUMNS = [
    "id", "label", "statement", "subject", "speaker",
    "speaker_job", "state", "party", "barely_true_count",
    "false_count", "half_true_count", "mostly_true_count",
    "pants_on_fire_count", "context"
]

TRAIN_URL = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/train.tsv"
TEST_URL  = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/test.tsv"
VALID_URL = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/valid.tsv"

train_df = pd.read_csv(TRAIN_URL, sep='\t', header=None, names=COLUMNS)
test_df  = pd.read_csv(TEST_URL,  sep='\t', header=None, names=COLUMNS)
valid_df = pd.read_csv(VALID_URL, sep='\t', header=None, names=COLUMNS)

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")
train_df[['label', 'statement']].head(5)

Train: 10240 | Valid: 1284 | Test: 1267


,label,statement
0,false,Says the Annies List political group supports ...
1,half-true,When did the decline of coal start? It started...
2,mostly-true,"Hillary Clinton agrees with John McCain ""by vo..."
3,false,Health care reform legislation is likely to ma...
4,half-true,The economic turnaround started at the end of ...


In [13]:
FAKE_LABELS = {"pants-fire", "false", "barely-true"}
REAL_LABELS = {"half-true", "mostly-true", "true"}

def binarize(label):
    if label in FAKE_LABELS: return 0
    if label in REAL_LABELS: return 1
    return None

for df in [train_df, test_df, valid_df]:
    df["binary_label"] = df["label"].apply(binarize)

train_df = train_df.dropna(subset=["binary_label"])
test_df  = test_df.dropna(subset=["binary_label"])
valid_df = valid_df.dropna(subset=["binary_label"])

print("Label distribution (train):")
print(train_df["binary_label"].value_counts().rename({0: "FAKE", 1: "REAL"}))

Label distribution (train):
binary_label
REAL    5752
FAKE    4488
Name: count, dtype: int64


In [14]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in STOP_WORDS]
    return " ".join(tokens)

train_df["clean"] = train_df["statement"].apply(preprocess)
test_df["clean"]  = test_df["statement"].apply(preprocess)
valid_df["clean"] = valid_df["statement"].apply(preprocess)

X_train, y_train = train_df["clean"], train_df["binary_label"].astype(int)
X_test,  y_test  = test_df["clean"],  test_df["binary_label"].astype(int)

print("Sample cleaned statement:")
print(X_train.iloc[0])

Sample cleaned statement:
says annies list political group supports thirdtrimester abortions demand


In [15]:
# TF-IDF + Logistic Regression
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"))
])
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

# TF-IDF + Naive Bayes (baseline)
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
    ("clf",   MultinomialNB(alpha=0.1))
])
nb_pipeline.fit(X_train, y_train)
nb_preds = nb_pipeline.predict(X_test)

print("=" * 50)
print("LOGISTIC REGRESSION")
print(f"Accuracy: {accuracy_score(y_test, lr_preds)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, lr_preds, average='weighted'):.4f}")
print(classification_report(y_test, lr_preds, target_names=["FAKE", "REAL"]))

print("=" * 50)
print("NAIVE BAYES (baseline)")
print(f"Accuracy: {accuracy_score(y_test, nb_preds)*100:.2f}%")
print(f"F1 Score: {f1_score(y_test, nb_preds, average='weighted'):.4f}")

LOGISTIC REGRESSION
Accuracy: 61.33%
F1 Score: 0.5982
              precision    recall  f1-score   support

        FAKE       0.58      0.40      0.47       553
        REAL       0.63      0.78      0.69       714

    accuracy                           0.61      1267
   macro avg       0.60      0.59      0.58      1267
weighted avg       0.61      0.61      0.60      1267

NAIVE BAYES (baseline)
Accuracy: 61.09%
F1 Score: 0.6015


In [16]:
headlines = [
    "Scientists confirm vaccines cause autism in children",
    "NASA announces new mission to explore Jupiter's moons",
    "Government secretly putting chemicals in drinking water",
    "India launches record number of satellites in single mission",
]

print("Custom Headline Predictions:\n")
for h in headlines:
    clean = preprocess(h)
    pred  = lr_pipeline.predict([clean])[0]
    prob  = lr_pipeline.predict_proba([clean])[0]
    label = " REAL" if pred == 1 else " FAKE"
    print(f"{label} ({max(prob)*100:.1f}%) — {h}")

Custom Headline Predictions:

 FAKE (61.1%) — Scientists confirm vaccines cause autism in children
 REAL (51.8%) — NASA announces new mission to explore Jupiter's moons
 FAKE (50.4%) — Government secretly putting chemicals in drinking water
 REAL (70.1%) — India launches record number of satellites in single mission
